# JupyterLite
## Interactive Python in the browser — no server required

A static site you can host on GitHub Pages, GitLab Pages, or an S3 bucket.
Everything you see runs **in your browser**, powered by WebAssembly.

*Andre Geldenhuis*

### How to drive this deck

- This **is** a Jupyter notebook, presented as slides with `jupyterlab-deck`.
- Before presenting: **Run → Run All Cells** so the live demos are ready.
- Toggle deck mode with the **slideshow / deck** button in the toolbar (or the command palette: *Start Deck*).
- Arrow keys move between slides; code cells stay live — you can run them mid-talk.

# Two ways to put Python on the web

| Dynamic (server-backed) | Static (in the browser) |
|---|---|
| JupyterHub, Binder, Voila | **JupyterLite** |
| Shiny, Streamlit, Dash | nbconvert, Voici |
| Code runs on **a server** | Code runs on **the visitor's machine** |
| You manage compute, scaling, auth, uptime | Just static files behind a CDN |
| Costs scale with users | Cost ~flat; scales for free |

## Why static is appealing

- **No server to run** — no JupyterHub to babysit, no Shiny server, no bill that grows with traffic.
- **Trivial hosting** — GitHub/GitLab Pages, S3, any static host. GitHub Pages gives ~100 GB/month free.
- **Safe by construction** — visitors run code in *their own* sandbox, not on your machine.
- **Reproducible & shareable** — send a URL; it just works, no installs.

## The trade-offs (be honest)

- First load downloads the Python runtime + packages → **slow cold start** (seconds to tens of seconds).
- Limited to packages compiled for WebAssembly.
- Runs in the browser sandbox: **no real filesystem, no threads/sockets** the way native Python has them.
- Heavy compute is bounded by the *visitor's* laptop, not a beefy server.

Great for **demos, teaching, and tinkerable data**. Not a replacement for a real compute backend.

# What is WebAssembly?

**WebAssembly (Wasm)** is a portable binary instruction format that runs in every modern browser
at near-native speed, inside the same security sandbox as JavaScript.

- Designed as a **compilation target** — you write C / C++ / Rust (or a whole language runtime) and compile *to* Wasm.
- Fast, sandboxed, and language-agnostic.
- The key move: you can compile **CPython itself** to Wasm → Python running client-side.

## Why Wasm makes JupyterLite possible

- Browser already ships a Wasm engine — nothing to install.
- The Python interpreter + your packages are shipped as Wasm/data files.
- The notebook UI talks to that in-browser kernel exactly like it would talk to a remote one.

> No server in the loop — the "kernel" is a tab in your browser.

# Two Python-in-Wasm kernels: Pyodide vs xeus

| | **Pyodide** | **xeus-python** |
|---|---|---|
| What | CPython compiled to Wasm + JS glue | `xeus` kernel + CPython via emscripten-forge |
| Packages | `micropip` / Pyodide package set (PyPI-ish) | conda-style, from **emscripten-forge** |
| Startup | Larger, slower cold start | Leaner, generally **faster to load** |
| Maturity | Older, huge ecosystem | Newer, growing fast |
| Default in JupyterLite | Yes (pyodide kernel) | Opt-in via `jupyterlite-xeus` |

## This talk uses **xeus**

```yaml
# environment.yml  — the in-browser runtime
channels:
  - https://repo.mamba.pm/emscripten-forge
  - conda-forge
dependencies:
  - python=3.11
  - xeus-python
  - ipycanvas
  - pandas
  - bqplot
```

You declare an `environment.yml`, `jupyterlite-xeus` builds a Wasm environment from it — the same mental model as conda, just compiled for the browser.

# Live demos

Everything from here runs **entirely in your browser**.
No server. No install. Just a URL.

## Demo 1 — drawing with `ipycanvas`

A live xeus-python kernel drawing to an HTML canvas. Run the cell:

In [ ]:
from math import pi
from ipycanvas import Canvas

canvas = Canvas(width=1600, height=1200, layout=dict(width="100%"))

canvas.fill_style = "#8ee05e"
canvas.fill_rect(0, 0, canvas.width, canvas.height)

canvas.fill_style = "#f5f533"
canvas.fill_circle(canvas.width / 2.0, canvas.height / 2.0, 500)

canvas.stroke_style = "black"
canvas.line_width = 30
canvas.stroke_circle(canvas.width / 2.0, canvas.height / 2.0, 500)

canvas.fill_style = "black"
canvas.fill_circle(canvas.width / 2.7, canvas.height / 3.0, 100)  # Right eye
canvas.stroke_arc(canvas.width / 2.0, canvas.height / 2.0, 400, 0, pi, False)  # Mouth
canvas.stroke_arc(
    canvas.width - canvas.width / 2.7, canvas.height / 2.7, 100, 0, pi, True
)  # Left eye

canvas

## Demo 2 — a tinkerable data explorer

`pandas` + `bqplot` + Jupyter widgets — pick indicators, drag the year slider,
and the scatter plot updates live. **Your audience can change the question, not just watch the answer.**

In [ ]:
# --- setup (hidden from the deck, runs on Run All) ---
import pandas as pd
import numpy as np
import ipywidgets as widgets

from bqplot import Figure, Scatter, Axis, LinearScale

EXPLANATION = """\
<div class="app-sidebar">
<p><em>Compare different development indicators.</em><p>
<p>Select indicators in the dropdowns, and use the slider to sub-select years.</p>
<p>Data and idea from the <a href="https://dash.plot.ly/getting-started-part-2">Plotly Dash docs</a>.</p>
</div>
"""


class App:

    def __init__(self, df):
        self._df = df
        available_indicators = self._df['Indicator Name'].unique()
        self._x_dropdown = self._create_indicator_dropdown(available_indicators, 0)
        self._y_dropdown = self._create_indicator_dropdown(available_indicators, 1)

        x_scale = LinearScale()
        y_scale = LinearScale()
        self._x_axis = Axis(scale=x_scale, label="X")
        self._y_axis = Axis(scale=y_scale, orientation="vertical", label="Y")
        self._scatter = Scatter(x=[], y=[], scales={"x": x_scale, "y": y_scale})
        self._figure = Figure(
            marks=[self._scatter], axes=[self._x_axis, self._y_axis],
            layout=dict(width="99%"), animation_duration=1000,
        )

        self._year_slider, year_slider_box = self._create_year_slider(
            min(df['Year']), max(df['Year'])
        )
        _app_container = widgets.VBox([
            widgets.HBox([self._x_dropdown, self._y_dropdown]),
            self._figure,
            year_slider_box,
        ], layout=widgets.Layout(align_items='center', flex='3 0 auto'))
        self.container = widgets.VBox([
            widgets.HBox([
                _app_container,
                widgets.HTML(EXPLANATION, layout=widgets.Layout(margin='0 0 0 2em')),
            ])
        ], layout=widgets.Layout(flex='1 1 auto', margin='0 auto 0 auto', max_width='1024px'))
        self._update_app()

    @classmethod
    def from_csv(cls, path):
        return cls(pd.read_csv(path))

    def _create_indicator_dropdown(self, indicators, initial_index):
        dropdown = widgets.Dropdown(options=indicators, value=indicators[initial_index])
        dropdown.observe(self._on_change, names=['value'])
        return dropdown

    def _create_year_slider(self, min_year, max_year):
        year_slider_label = widgets.Label('Year range: ')
        year_slider = widgets.IntRangeSlider(
            min=min_year, max=max_year,
            layout=widgets.Layout(width='500px'), continuous_update=False,
        )
        year_slider.observe(self._on_change, names=['value'])
        return year_slider, widgets.HBox([year_slider_label, year_slider])

    def _on_change(self, _):
        self._update_app()

    def _update_app(self):
        x_indicator = self._x_dropdown.value
        y_indicator = self._y_dropdown.value
        year_range = self._year_slider.value
        with self._scatter.hold_sync():
            df = self._df[self._df['Year'].between(*year_range)].dropna()
            self._x_axis.label = x_indicator
            self._y_axis.label = y_indicator
            self._scatter.default_opacities = [0.2]
            self._scatter.x = df[df['Indicator Name'] == x_indicator]['Value']
            self._scatter.y = df[df['Indicator Name'] == y_indicator]['Value']

In [ ]:
app = App.from_csv("./indicators.csv")
app.container

# Where this shines in research

- **Reproducible explorers** — ship the data *and* the controls; reviewers poke at it, no setup.
- **Supplementary material** — link a live figure from a paper instead of a static PNG.
- **Lightweight teaching** — students open a URL and start coding; no installs, no "works on my machine".
- **Show, don't tell** — let people change parameters and *see* how results move.

# How the site gets built: CI/CD

You never build it by hand. A pipeline does it on every push:

1. Push notebooks to the repo.
2. CI installs the build env + `jupyter lite build --contents content`.
3. The resulting static `dist/` is published to Pages.

Works the same way on **GitHub Actions → GitHub Pages** and **GitLab CI → GitLab Pages** (or upload `dist/` to S3).

## The GitHub Actions shape

```yaml
- uses: mamba-org/setup-micromamba@v2
  with:
    environment-file: build-environment.yml

- name: Build the JupyterLite site
  run: jupyter lite build --contents content --output-dir dist

- uses: actions/upload-pages-artifact@v3
  with:
    path: ./dist
# ... then actions/deploy-pages@v4
```

Two environments: **build-environment.yml** (the builder) and **environment.yml** (the in-browser runtime).

# Practical advice (the stuff that bites you)

### "I pushed a fix but the site looks unchanged!"
- The Action takes a minute or two — check it actually **finished green** first.
- Then it's **caching**: both the CDN and your browser cache aggressively, and JupyterLite
  registers a **service worker** that caches assets.
- Fixes: **hard refresh** (Cmd/Ctrl + Shift + R), an **incognito window**, or bump a version / wait out the cache TTL.

## A dev workflow without running anything locally

You don't need a local server or even a local Python env:

1. Open your **deployed** JupyterLite site — it's a full JupyterLab in the browser.
2. Edit / experiment there, then **download** the changed notebook.
3. Commit it (drag-drop in the GitHub web UI, or `git` if you prefer).
4. Push → the Action rebuilds → hard-refresh to see it.

## Optional: build it locally with `micromamba`

Faster loops than push-and-wait. `micromamba` is a single static binary — no install, no admin.

```bash
# 1. Grab the binary (macOS arm64 shown; pick your platform)
curl -Ls https://micro.mamba.pm/api/micromamba/osx-arm64/latest | tar -xvj bin/micromamba

# 2. Create the BUILD environment (the builder, not the in-browser runtime)
micromamba create -y -n lite-build -f build-environment.yml

# 3. Build the site  —  micromamba MUST be on PATH:
#    jupyter-lite-xeus shells out to it to build the Wasm kernel
micromamba run -n lite-build jupyter lite build --contents content --output-dir dist

# 4. Serve it
python -m http.server -d dist 8000   # → http://localhost:8000/lab
```

# Try it / steal it

- **This deck & demos:** the repo you're looking at
- **Python image-manipulation teaching session** — [live](https://andre-geldenhuis.github.io/python-image-manipulation-session/) · [github](https://github.com/andre-geldenhuis/python-image-manipulation-session)
- **Voici demo** — [live](https://voila-dashboards.github.io/voici-demo) · [github](https://github.com/voila-dashboards/voici-demo)
- **JupyterLite docs** — jupyterlite.readthedocs.io
- **emscripten-forge** (xeus packages) — emscripten-forge.org

# Thanks — questions?

Everything you saw ran in your browser, served as static files.

No server. No install. Just a URL.